In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 2. Install dependencies (do NOT touch torch — Colab already has it)
!pip install -q transformers datasets scikit-learn

In [ ]:
# 3. Imports
import os
import json
import numpy as np
import torch
from torch.nn import CrossEntropyLoss
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from datasets import Dataset
import datasets
datasets.config.TORCHVISION_AVAILABLE = False   # fixes VideoReader ImportError

In [ ]:
# 4. Paths (update DRIVE_FOLDER to your actual mounted path)
DRIVE_FOLDER = "/content/drive/MyDrive/FIRE-2026"  # <-- update this
TRAIN_PATH = f"{DRIVE_FOLDER}/train_subtask1.json"
DEV_PATH = f"{DRIVE_FOLDER}/dev_subtask1.json"
OUTPUT_DIR = "/content/drive/MyDrive/FIRE-2026/FINAL_MODEL_1"

os.environ["TENSORBOARD_LOGGING_DIR"] = f"{OUTPUT_DIR}/logs"

In [ ]:
# ============================================
# 5. DATA PREP
# ============================================

with open(TRAIN_PATH, "r", encoding="utf-8") as f:
    train_data = json.load(f)
with open(DEV_PATH, "r", encoding="utf-8") as f:
    dev_data = json.load(f)

label2id = {"SUPPORTS": 0, "REFUTES": 1}
id2label = {0: "SUPPORTS", 1: "REFUTES"}

def to_hf_dataset(data):
    return Dataset.from_dict({
        "claim": [x["Text"] for x in data],
        "evidence": [x["Evidence"] for x in data],
        "label": [label2id[x["Label"]] for x in data],
    })

In [ ]:
# 6. Tokenizer & Model
MODEL_NAME = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id
)

def tokenize_fn(examples):
    return tokenizer(
        examples["claim"],
        examples["evidence"],
        truncation=True,
        max_length=192,
        padding="max_length",
    )

# Build + tokenize datasets
train_dataset = to_hf_dataset(train_data).map(tokenize_fn, batched=True)
dev_dataset = to_hf_dataset(dev_data).map(tokenize_fn, batched=True)

# Drop raw text columns
train_dataset = train_dataset.remove_columns(["claim", "evidence"])
dev_dataset = dev_dataset.remove_columns(["claim", "evidence"])

# Set torch format
train_dataset.set_format("torch")
dev_dataset.set_format("torch")

# Sanity check
assert "input_ids" in train_dataset.column_names, "Tokenization failed!"
print("✅ Columns:", train_dataset.column_names)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

Map:   0%|          | 0/4536 [00:00<?, ? examples/s]

Map:   0%|          | 0/1134 [00:00<?, ? examples/s]

✅ Columns: ['label', 'input_ids', 'token_type_ids', 'attention_mask']


In [ ]:
# ============================================
# 7. Class weights (auto-computed from data)
# ============================================

labels = [x["label"].item() for x in train_dataset]
class_counts = np.bincount(labels)
class_weights = torch.tensor(
    [len(labels) / (2 * c) for c in class_counts], dtype=torch.float
)
print("Class weights:", class_weights)

Class weights: tensor([0.7461, 1.5160])


In [ ]:
# ============================================
# 8. Custom Trainer with weighted loss
# ============================================

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = CrossEntropyLoss(weight=class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [ ]:
# ============================================
# 9. Metrics (overall + per-class)
# ============================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average="macro")

    precision, recall, f1_per_class, _ = precision_recall_fscore_support(
        labels, preds, labels=[0, 1], zero_division=0
    )

    return {
        "accuracy": acc,
        "f1": f1_macro,
        "precision_supports": precision[0],
        "recall_supports": recall[0],
        "f1_supports": f1_per_class[0],
        "precision_refutes": precision[1],
        "recall_refutes": recall[1],
        "f1_refutes": f1_per_class[1],
    }

In [ ]:
# ============================================
# 10. Training Arguments (BEST hyperparameters from Optuna search)
# ============================================

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=1.7991465194475932e-05,
    num_train_epochs=16,
    weight_decay=0.0764751706565559,
    warmup_ratio=0.026509480810828625,
    max_grad_norm=1.2004964752527867,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=50,
    report_to="none",
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
# ============================================
# 11. Trainer
# ============================================

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [ ]:
# ============================================
# 12. Train
# ============================================

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision Supports,Recall Supports,F1 Supports,Precision Refutes,Recall Refutes,F1 Refutes
1,0.692097,0.673863,0.731041,0.690368,0.771144,0.836707,0.802589,0.633333,0.531807,0.578147
2,0.638831,0.615735,0.768078,0.681595,0.742886,0.986505,0.847536,0.933333,0.356234,0.515654
3,0.541064,0.525182,0.804233,0.773791,0.820766,0.896086,0.856774,0.763077,0.631043,0.690808
4,0.450424,0.476167,0.831570,0.809626,0.853470,0.896086,0.874259,0.783708,0.709924,0.744993
5,0.386122,0.460240,0.817460,0.800234,0.867769,0.850202,0.858896,0.727941,0.755725,0.741573
6,0.268048,0.471786,0.825397,0.808373,0.871409,0.859649,0.865489,0.741935,0.760814,0.751256
7,0.248123,0.495761,0.794533,0.781495,0.879104,0.794872,0.834869,0.672414,0.793893,0.728121
8,0.206022,0.584846,0.813051,0.799183,0.882779,0.823212,0.851955,0.704289,0.793893,0.746411
9,0.198999,0.593806,0.834215,0.811976,0.853129,0.901484,0.876640,0.792023,0.707379,0.747312
10,0.167928,0.613931,0.823633,0.807956,0.877266,0.848853,0.862826,0.731415,0.776081,0.753086


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

... (8 outputs left out)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2272, training_loss=0.2890164910907477, metrics={'train_runtime': 1161.5181, 'train_samples_per_second': 62.484, 'train_steps_per_second': 1.956, 'total_flos': 7160830482677760.0, 'train_loss': 0.2890164910907477, 'epoch': 16.0})

In [ ]:
# ============================================
# 13. Save final model + tokenizer to Drive
# ============================================

FINAL_MODEL_DIR = f"{OUTPUT_DIR}/final_model"
trainer.save_model(FINAL_MODEL_DIR)
tokenizer.save_pretrained(FINAL_MODEL_DIR)
print("✅ Training complete. Model saved at:", FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Training complete. Model saved at: /content/drive/MyDrive/FIRE-2026/FINAL_MODEL_1/final_model


In [ ]:
# ============================================
# 14. Final Evaluation
# ============================================

metrics = trainer.evaluate()
print(metrics)

Training Loss,Validation Loss,Epoch,Accuracy,F1,Precision Supports,Recall Supports,F1 Supports,Precision Refutes,Recall Refutes,F1 Refutes
0.090869,0.460240,16,0.817460,0.800234,0.867769,0.850202,0.858896,0.727941,0.755725,0.741573


{'eval_loss': 0.46023958921432495, 'eval_accuracy': 0.8174603174603174, 'eval_f1': 0.8002343696146688, 'eval_precision_supports': 0.8677685950413223, 'eval_recall_supports': 0.8502024291497976, 'eval_f1_supports': 0.8588957055214724, 'eval_precision_refutes': 0.7279411764705882, 'eval_recall_refutes': 0.7557251908396947, 'eval_f1_refutes': 0.7415730337078652}


In [ ]:
# ============================================
# 15. Inference on Test Set (test_subtask1.json)
# ============================================

import torch.nn.functional as F

TEST_PATH = f"{DRIVE_FOLDER}/test_subtask1.json"

with open(TEST_PATH, "r", encoding="utf-8") as f:
    test_data = json.load(f)

def to_hf_test_dataset(data):
    return Dataset.from_dict({
        "id": [x["ID"] for x in data],
        "claim": [x["Text"] for x in data],
        "evidence": [x["Evidence"] for x in data],
    })

test_dataset_raw = to_hf_test_dataset(test_data)

def tokenize_test_fn(examples):
    return tokenizer(
        examples["claim"],
        examples["evidence"],
        truncation=True,
        max_length=192,
        padding="max_length",
    )

test_dataset = test_dataset_raw.map(tokenize_test_fn, batched=True)
test_dataset_for_model = test_dataset.remove_columns(["id", "claim", "evidence"])
test_dataset_for_model.set_format("torch")

# Load the best saved model explicitly (ensures we use the final trained model)
inference_model = AutoModelForSequenceClassification.from_pretrained(FINAL_MODEL_DIR)
inference_model.eval()
inference_model.to("cuda" if torch.cuda.is_available() else "cpu")

device = next(inference_model.parameters()).device

predictions = []
with torch.no_grad():
    for i in range(len(test_dataset_for_model)):
        item = test_dataset_for_model[i]
        input_ids = item["input_ids"].unsqueeze(0).to(device)
        attention_mask = item["attention_mask"].unsqueeze(0).to(device)
        token_type_ids = item["token_type_ids"].unsqueeze(0).to(device)

        outputs = inference_model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        probs = F.softmax(outputs.logits, dim=-1)
        pred_label_id = torch.argmax(probs, dim=-1).item()
        confidence = probs[0][pred_label_id].item()

        predictions.append({
            "ID": test_data[i]["ID"],
            "Text": test_data[i]["Text"],
            "Predicted_Label": id2label[pred_label_id],
            "Confidence": round(confidence, 4),
        })

# Print results
for p in predictions:
    print(f"{p['ID']} | {p['Predicted_Label']} (conf={p['Confidence']}) | {p['Text'][:60]}...")

# Save predictions to Drive
PRED_OUTPUT_PATH = f"{OUTPUT_DIR}/test_predictions.json"
with open(PRED_OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

print(f"\n✅ Predictions saved to: {PRED_OUTPUT_PATH}")

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

S1/4530 | SUPPORTS (conf=0.8339) | #IndiaTVSamvaad: भाजपा अध्यक्ष अमित शाह ने कहा, 'NDA 2019 मे...
S1/4968 | SUPPORTS (conf=0.6845) | 'सुधार की संभावना से इंकार नहीं किया जा सकता': सुप्रीम कोर्ट...
S1/3419 | SUPPORTS (conf=0.8229) | हरियाणा में टीके न लगवाने वालों को सार्वजनिक जगहों पर एंट्री...
S1/4576 | SUPPORTS (conf=0.8014) | स्वीकृत मेडिकल पद्धतियों का पालन करने वाले डॉक्टर सर्जरी के ...
S1/1133 | SUPPORTS (conf=0.8407) | सुप्रीम कोर्ट ने मॉब लिंचिंग के खिलाफ जनहित याचिका में जवाबी...
S1/6482 | REFUTES (conf=0.9023) | राज्यपाल को सौंपी लिस्ट में भाजपा विधायकों के फ़र्ज़ी हस्ताक...
S1/6301 | REFUTES (conf=0.9022) | हीरा गोल्ड घोटाला: सुप्रीम कोर्ट ने ED को तीन संपत्तियों की ...
S1/6200 | REFUTES (conf=0.9023) | सिविल विवादों को आपराधिक मामलों में बदलने की गलत प्रथा केवल ...
S1/1169 | REFUTES (conf=0.9023) | “Omicron” के इलाज का खर्च Coronavirus Health Insurance Polic...
S1/5520 | SUPPORTS (conf=0.8118) | चीफ जस्टिस पर महाभियोग मामले में विपक्ष की 'जीत' तय, जानिए क...
S1/3590 | SUPP